# __Data Cleaning__
1. Delete whitespace in column names.
2. Replace NaN value with median of dataset.
3. Drop duplicate rows.
4. Replace +-inf value with median of dataset.
5. Rebalance dataset using SMOTE method.

# Import libs

In [ ]:
!pip install fastparquet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

import fastparquet
import seaborn as sns
import pyarrow.parquet as pq
from sklearn.preprocessing import MinMaxScaler
import os

import glob
from tqdm import tqdm

In [ ]:
path = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/Syn.csv'
df = pd.read_csv(path)
print('Data Shape:', df.shape)
print(df.head())
print(df.info())

In [ ]:
df.columns = df.columns.str.strip()
label = df['Label']

df.drop(columns=['Unnamed: 0', "Source Port", "Destination Port", "Protocol"], inplace=True)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df = df[numeric_cols].copy()

In [ ]:
has_nan = df.isnull().values.any()
print(f"Contains NaN?: {has_nan}")
if has_nan:
  count_nan = df.isnull().sum().sum()
  print(f"Have {count_nan} NaN")
  df.fillna(df.median(), inplace=True)

has_dup = df.duplicated().values.any()
print(f"Contains duplicate rows?: {has_dup}")
if has_dup:
  count_dup = df.duplicated().sum()
  print(f"Have duplicate {count_dup} rows")
  index_dup_rows = df.index[df.duplicated()]
  label.drop(index=index_dup_rows, inplace=True)
  df.drop_duplicates(inplace = True)

has_inf = np.isinf(df).any().any()
print(f"Contains inf rows?: {has_inf}")
if has_inf:
    count_inf = np.isinf(df).sum().sum()
    print(f"Have inf {count_inf} rows. Then replacing inf with NaN and filling NaN with median.")
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(df.median(), inplace=True)

In [ ]:
non_nan_labels_mask = label.notna()

df_for_smote = df[non_nan_labels_mask].copy()
label_for_smote = label[non_nan_labels_mask].copy()

count_class = label_for_smote.value_counts()
plt.bar(count_class.index, count_class.values)
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution (Before SMOTE)')
plt.xticks(count_class.index, [f'Class {i}' for i in count_class.index])
plt.show()

smote = SMOTE(sampling_strategy='not majority')
X_res, y_res = smote.fit_resample(df_for_smote, label_for_smote)
print("After SMOTE:\n", y_res.value_counts())

In [ ]:
df_resampled = pd.DataFrame(X_res, columns=df_for_smote.columns)
df_resampled['Label'] = y_res

print('Shape of resampled DataFrame:', df_resampled.shape)
print(df_resampled.head())

output_path = 'resampled_DrDoS_SSDP.csv'
df_resampled.to_csv(output_path, index=False)
print(f"Resampled data exported to '{output_path}'")

---

#CSV to parquet

In [ ]:
trainList = ['DrDoS_DNS',
             'DrDoS_NTP',
             'DrDoS_NetBIOS', 'DrDoS_SNMP',
             'DrDos_SSDP',
             'DrDoS_LDAP',
             'DrDoS_MSSQL',
             'DrDoS_UDP',
             'Syn',
             'UDPLag']
testList = ['LDAP', 'MSSQL', 'NetBIOS', 'Portmap', 'Syn', 'UDP', 'UDPLag']

# for fileName in testList:
for fileName in trainList:
  csv_path = '/content/drive/MyDrive/Dataset492/1-12 (training set)/' + fileName + '.csv'
  parquet_path = '/content/drive/MyDrive/Dataset492/1-12 (training set)/' + fileName + '.parquet'
  # csv_path = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/' + fileName + '.csv'
  # parquet_path = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/' + fileName + '.parquet'
  chunk_size = 100000

  parquet_schema = None
  writer = None

  for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):

      table = pa.Table.from_pandas(chunk)

      if i == 0:
          parquet_schema = table.schema
          writer = pq.ParquetWriter(parquet_path, parquet_schema, compression='snappy')

      writer.write_table(table)

  if writer:
      writer.close()
      print(fileName, "Complete!")

---

# Drop unuse table

In [ ]:
trainList = [
             'DrDoS_DNS',
             'DrDoS_NTP',
             'DrDoS_NetBIOS', 'DrDoS_SNMP',
             'DrDos_SSDP',
             'DrDoS_LDAP',
             'DrDoS_MSSQL',
             'DrDoS_UDP',
             'Syn',
             'UDPLag']

testList = ['LDAP', 'MSSQL', 'NetBIOS', 'Portmap', 'Syn', 'UDP', 'UDPLag']

unuse_column = ['Bwd PSH Flags',
'Fwd URG Flags',
'Bwd URG Flags',
'FIN Flag Count',
'PSH Flag Count',
'ECE Flag Count',
'Fwd Avg Bytes/Bulk',
'Fwd Avg Packets/Bulk',
'Fwd Avg Bulk Rate',
'Bwd Avg Bytes/Bulk',
'Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate']

for fileName in testList:
  parquet_path = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/parquet/' + fileName + '.parquet'
  df = pd.read_parquet(parquet_path, engine='pyarrow')
  df.drop(columns=unuse_column, inplace=True)
  print(fileName)
  print(df.info())
  df.to_parquet(parquet_path, engine='pyarrow')


In [ ]:
unuse_column = ['Bwd PSH Flags',
'Fwd URG Flags',
'Bwd URG Flags',
'FIN Flag Count',
'PSH Flag Count',
'ECE Flag Count',
'Fwd Avg Bytes/Bulk',
'Fwd Avg Packets/Bulk',
'Fwd Avg Bulk Rate',
'Bwd Avg Bytes/Bulk',
'Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate']

df.drop(columns=unuse_column, inplace=True)

# 1. Drop the 'Label' column for correlation calculation
df_corr = df.drop(columns=['Label'])

# 2. Calculate the Spearman correlation matrix
corr_matrix = df_corr.corr(method='spearman')

# 3. Create a heatmap using seaborn
plt.figure(figsize=(20, 18)) # Adjust figure size for better readability
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title(f'Spearman Correlation Matrix {len(df_corr.columns.tolist())} Features(excluding Label)', fontsize=16)
plt.xticks(rotation=90)
plt.yticks(rotation=0)

# 6. Display the plot
plt.show()

---

# Verify the consistency of the columns in all cleaned datasets.


In [ ]:
def check_parquet_schema(folder_path):
    files = glob.glob(os.path.join(folder_path, "*.parquet"))

    if not files:
        print("No Parquet files found in this folder.")
        return

    print(f"Found {len(files)} files. Starting verification...\n")

    first_file = files[0]
    try:
        ref_schema = pq.read_schema(first_file)
        print(f"Reference file: {os.path.basename(first_file)}")
        print(f"   - Total Columns: {len(ref_schema.names)}")
    except Exception as e:
        print(f"Error reading the first file: {e}")
        return

    mismatch_files = []

    for f in files[1:]:
        filename = os.path.basename(f)
        try:
            current_schema = pq.read_schema(f)

            if not ref_schema.equals(current_schema):
                print(f"Mismatch found: {filename}")

                ref_cols = set(ref_schema.names)
                cur_cols = set(current_schema.names)

                missing = ref_cols - cur_cols
                extra = cur_cols - ref_cols

                if missing: print(f"    - Missing columns: {missing}")
                if extra: print(f"    - Extra columns: {extra}")
                if not missing and not extra: print(f"    - Column names match, but order or data types differ.")

                mismatch_files.append(filename)

        except Exception as e:
            print(f"Error reading file {filename}: {e}")

    print("\n" + "="*30)
    if not mismatch_files:
        print("All files have matching structures (Index & Columns).")
    else:
        print(f"Found {len(mismatch_files)} problematic files.")

folder_path = '/content/drive/MyDrive/Dataset492/1-12 (training set)/parquet'
print("\n" + "="*30)
print("       TRAINING FOLDER\n")
check_parquet_schema(folder_path)
folder_path = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/parquet'
print("\n" + "="*30)
print("       TESTING FOLDER\n")
check_parquet_schema(folder_path)

---

# Features Selection
Steps:
1. Read all parquet files
2. Calculate Spearman correlation matrix
3. Drop features with correlation > 0.85
4. Map target labels to integers
5. Normalize using MinMaxScaler

read file

In [ ]:
file_name = 'DrDoS_DNS'
input_path = '/content/drive/MyDrive/Dataset492/1-12 (training set)/parquet/' + file_name + '.parquet'
output_path = '/content/drive/MyDrive/Dataset492/1-12 (training set)/selected_feature/' + file_name + '.parquet'

df = pd.read_parquet(input_path, engine='pyarrow')

# TODO: map label

df_exclude_label = df.drop(columns=['Label'])

scaler = MinMaxScaler()
scaler.fit(df_exclude_label)

corr_matrix = df_exclude_label.corr(method='spearman').abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

X_reduced = df_exclude_label.drop(columns=to_drop)

corr_matrix = X_reduced.corr(method='spearman').abs()
# show image
plt.figure(figsize=(20, 15))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, fmt='.2f', vmin=0, vmax=1)
plt.title(f"Spearman Correlation Matrix {len(X_reduced.columns.to_list())} Features (After Dropping Features > 0.8)")
plt.show()

# TODO: write file

read chuck

In [ ]:
# List of parquet files to process
parquet_files = ['DrDoS_DNS']
# ,'DrDoS_LDAP', 'DrDoS_MSSQL', 'DrDoS_NTP',
#              'DrDoS_NetBIOS', 'DrDoS_SNMP', 'DrDoS_UDP', 'DrDos_SSDP',
#              'Syn', 'UDPLag']

parquet_folder = '/content/drive/MyDrive/Dataset492/1-12 (training set)/parquet'
output_folder = '/content/drive/MyDrive/Dataset492/1-12 (training set)/selected_feature'
os.makedirs(output_folder, exist_ok=True)

print(f"Processing {len(parquet_files)} parquet files from {parquet_folder}")
print(f"Files to process: {parquet_files}")

# ==============================================================================
# ฟังก์ชันสำหรับประมวลผลข้อมูลเป็นชุด
# ==============================================================================
def process_data_chunk(chunk_df, scaler, label_mapping, initial_features, to_drop):
    """ทำการประมวลผล (ลดคอลัมน์, เข้ารหัส Label, และปรับมาตราส่วน) สำหรับ DataFrame ชุดเล็ก"""

    # 1. แยก Features (X) และ Target (y)
    y_chunk = chunk_df['Label']
    X_chunk = chunk_df.drop(columns=['Label'])

    # 2. กรองเฉพาะคอลัมน์ที่เป็นตัวเลขและคอลัมน์ที่ลดแล้ว
    # ตรวจสอบว่าคอลัมน์ที่ต้องลดออกยังคงอยู่ในชุดข้อมูลปัจจุบันหรือไม่
    X_numeric_chunk = X_chunk[initial_features]
    X_reduced_chunk = X_numeric_chunk.drop(columns=to_drop, errors='ignore')

    # 3. เข้ารหัส Target (y)
    y_encoded_chunk = y_chunk.map(label_mapping)

    # 4. ปรับมาตราส่วน (Normalize)
    # ใช้ scaler ที่ 'fit' มาแล้วจากชุดข้อมูลแรก
    X_normalized_chunk = scaler.transform(X_reduced_chunk)

    # 5. แปลงกลับเป็น DataFrame
    X_normalized_df = pd.DataFrame(
        X_normalized_chunk,
        columns=X_reduced_chunk.columns,
        index=X_reduced_chunk.index
    )

    # 6. รวมและคืนค่า DataFrame ที่ประมวลผลแล้ว
    df_processed_chunk = X_normalized_df.copy()
    df_processed_chunk['Label'] = y_encoded_chunk.values

    return df_processed_chunk

# ==============================================================================
# ลูปประมวลผลไฟล์
# ==============================================================================

for file_name in parquet_files:
    print(f"\n{'='*80}")
    print(f"Processing: {file_name}.parquet (Using Chunking)")
    print('='*80)

    file_path = os.path.join(parquet_folder, file_name+'.parquet')

    # ----------------------------------------------------
    # PHASE 1: อ่านชุดข้อมูลขนาดเล็กชุดแรกเพื่อกำหนดโมเดล (Correlation, Mapping, Scaler)
    # ----------------------------------------------------

    try:
        # ใช้ pyarrow.parquet เพื่ออ่านชุดข้อมูลแรก
        parquet_file = pq.ParquetFile(file_path)

        #  อ่านเฉพาะ 100,000 แถวแรกเพื่อคำนวณ Correlation และ Label Mapping (เพื่อประหยัด RAM)
        # หาก 100,000 แถวไม่เป็นตัวแทนที่ดี คุณอาจต้องเพิ่มขนาดตรงนี้
        first_chunk_size = 100000

        # อ่านชุดแรก
        first_chunk_table = next(parquet_file.iter_batches(batch_size=first_chunk_size))
        df_first_chunk = first_chunk_table.to_pandas()

        print(f"Loaded initial chunk (100k rows) for model fitting.")

    except Exception as e:
        print(f"Error loading initial chunk for {file_name}: {e}")
        continue

    # --- Step 1 & 4: แยก/กำหนด Target Label ---
    if 'Label' not in df_first_chunk.columns:
        print(f"Warning: 'Label' column not found in {file_name}")
        continue

    y_first = df_first_chunk['Label']
    X_first = df_first_chunk.drop(columns=['Label'])

    # Get numeric columns before dropping label
    initial_features = X_first.select_dtypes(include=[np.number]).columns.tolist()
    X_numeric_first = X_first[initial_features].copy()

    # Label Mapping
    unique_labels = y_first.unique()
    label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
    print(f"Label mapping determined: {label_mapping}")

    # --- Step 2 & 3: คำนวณ Correlation และกำหนดคอลัมน์ที่จะลด ---
    print("Calculating Spearman correlation on initial chunk...")
    corr_matrix = X_numeric_first.corr(method='spearman').abs()

    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    to_drop = [column for column in upper_triangle.columns
               if any(upper_triangle[column] > 0.8)]

    print(f"Features to drop based on correlation (>0.8): {len(to_drop)} features")
    print(f"Features remaining: {len(initial_features) - len(to_drop)}")

    X_reduced_first = X_numeric_first.drop(columns=to_drop)

    # --- Step 5: Fit MinMaxScaler ---
    print("Fitting MinMaxScaler on reduced initial chunk...")
    scaler = MinMaxScaler()
    scaler.fit(X_reduced_first)

    # ----------------------------------------------------
    # PHASE 2: อ่านและประมวลผลไฟล์ทั้งหมดเป็นชุด ๆ แล้วเขียนลงไฟล์ใหม่
    # ----------------------------------------------------

    output_path = os.path.join(output_folder, file_name + '_processed.parquet')

    # กำหนดขนาดชุดข้อมูลสำหรับการประมวลผลจริง (ใหญ่กว่าเพื่อประสิทธิภาพ)
    chunk_size = 1000000

    # Create a sample processed DataFrame to infer schema
    sample_processed_df = process_data_chunk(
        df_first_chunk,
        scaler,
        label_mapping,
        initial_features,
        to_drop
    )
    # Convert to PyArrow Table and extract schema
    sample_table = pa.Table.from_pandas(sample_processed_df, preserve_index=False)

    # เตรียม ParquetWriter สำหรับการเขียนผลลัพธ์
    # ใช้สคีมาของ DataFrame ที่ลดขนาดแล้ว (X_reduced_first + Label)
    processed_schema_writer = pq.ParquetWriter(output_path, sample_table.schema)

    print(f"\nStarting chunk processing (Chunk size: {chunk_size} rows)...")

    # ใช้อีกครั้งเพื่อเริ่มต้นการอ่านจากจุดเริ่มต้น
    parquet_file = pq.ParquetFile(file_path)
    total_chunks = parquet_file.num_row_groups

    # ใช้ tqdm เพื่อติดตามความคืบหน้า
    for batch in tqdm(parquet_file.iter_batches(batch_size=chunk_size),
                      total=total_chunks,
                      desc=f"Processing {file_name}"):

        chunk_df = batch.to_pandas()

        # ประมวลผลชุดข้อมูลปัจจุบัน
        df_processed_chunk = process_data_chunk(
            chunk_df,
            scaler,
            label_mapping,
            initial_features,
            to_drop
        )

        # เขียนชุดข้อมูลที่ประมวลผลแล้วลงไฟล์ Parquet ใหม่
        table_chunk = pa.Table.from_pandas(df_processed_chunk, preserve_index=False)
        processed_schema_writer.write_table(table_chunk)

    processed_schema_writer.close() # ปิด Writer เพื่อเขียนส่วนท้ายของไฟล์

    # ----------------------------------------------------
    # PHASE 3: สรุป
    # ----------------------------------------------------

    print(f"\nProcessed data exported to: {output_path}")
    # (ไม่สามารถหา shape สุดท้ายได้โดยตรงหากไม่โหลดทั้งหมด ต้องใช้ pyarrow อ่านเมตาเดตา)
    # ลองโหลดเมตาเดตาเพื่อดูจำนวนแถวที่แท้จริง
    try:
        final_file = pq.ParquetFile(output_path)
        final_rows = final_file.metadata.num_rows
        print(f"Final shape (estimated from metadata): ({final_rows}, {len(final_file.schema.names)})")
    except Exception as e:
        print(f"Could not read final file metadata: {e}")

    print(f"Completed processing {file_name}\n")

In [ ]:
file_path = "/content/drive/MyDrive/Dataset492/1-12 (training set)/selected_feature/DrDoS_DNS_processed.parquet"
df = pd.read_parquet(file_path, engine='pyarrow')
df_exclude_label = df.drop(columns=['Label'])

corr_matrix = df_exclude_label.corr(method='spearman').abs()
plt.figure(figsize=(20, 15))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, fmt='.2f', vmin=0, vmax=1)
plt.title(f"Spearman Correlation Matrix {len(df_exclude_label.columns.to_list())} Features (After Dropping Features > 0.8)")
plt.show()

In [ ]:
parquet_files = [
    # 'DrDoS_DNS',
    # 'DrDoS_LDAP',
    # 'DrDoS_MSSQL',
    # 'DrDoS_NTP',
    # 'DrDoS_NetBIOS',
    # 'DrDoS_SNMP',
    # 'DrDoS_UDP',
    # 'DrDos_SSDP',
    # 'Syn',
    # 'UDPLag'
    # 'LDAP',
    'MSSQL',
    'NetBIOS',
    'Portmap',
    'Syn',
    'UDP',
    'UDPLag'
  ]

column_to_keep = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',
    'Fwd Packet Length Std',
    'Bwd Packet Length Min',
    'Bwd Packet Length Std',
    'Flow IAT Min',
    'Fwd IAT Total',
    'Fwd IAT Min',
    'Fwd PSH Flags',
    'Fwd Header Length',
    'Fwd Packets/s',
    'SYN Flag Count',
    'ACK Flag Count',
    'URG Flag Count',
    'CWE Flag Count',
    'Init_Win_bytes_forward',
    'act_data_pkt_fwd',
    'min_seg_size_forward',
    'Active Mean',
    'Idle Mean',
    'Label'
]

# parquet_folder = '/content/drive/MyDrive/Dataset492/1-12 (training set)/parquet'
# output_folder = '/content/drive/MyDrive/Dataset492/1-12 (training set)/selected_features'
parquet_folder = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/parquet'
output_folder = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/selected_features'

print(f"Processing {len(parquet_files)} parquet files from {parquet_folder}")


for file_name in parquet_files:
  print("\n" + "="*30)
  print(f"Dataset to process: {file_name}")
  df = pd.read_parquet(
      os.path.join(parquet_folder, file_name+'.parquet'),
      engine='pyarrow',
      columns=column_to_keep
  )
  print(df.info())

  df['Label'] = df['Label'].apply(lambda x: 1 if x != 'BENIGN' else 0)

  feature_cols = [col for col in df.columns if col != 'Label']
  scaler = MinMaxScaler()
  df[feature_cols] = scaler.fit_transform(df[feature_cols])

  output_filename = f"{file_name}_processed.parquet"
  output_path = os.path.join(output_folder, output_filename)
  df.to_parquet(output_path, engine='pyarrow')
  print(f"Saved: {output_filename} (Shape: {df.shape})")


---
# Train Model
---

## Import libs

In [ ]:
!pip install fastparquet
!pip install xgboost

In [ ]:
# Standard libraries
import os
import copy
import time

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.figure as fig
import matplotlib.cm as cm
import seaborn as sns

# File formats
import fastparquet
import pyarrow.parquet as pq

# Machine learning - preprocessing & metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, roc_curve
from sklearn.metrics import confusion_matrix

# Machine learning - models
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC

# save-load model
import joblib

## Training function

### RFC

In [ ]:
def train_random_forest(X_train, X_val, y_train, y_val):
    rf = RandomForestClassifier(
        n_jobs=-1,
        random_state=42,
        bootstrap=True,
        max_depth=20,
        max_features='sqrt',
        min_samples_split=10,
        n_estimators=100
        )

    print("[RandomForest] Training...")
    start = time.time()
    rf.fit(X_train, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = rf.predict(X_val)

    accuracy  = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1        = f1_score(y_val, y_pred, average='weighted', zero_division=0)

    return {
        "Model": "Random Forest",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Train Time (s)": train_time
    }, rf

### KNN

In [ ]:
def train_knn(X_train, X_val, y_train, y_val):
    knn = KNeighborsClassifier(n_neighbors=6, n_jobs=-1)

    start = time.time()
    knn.fit(X_train, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = knn.predict(X_val)

    accuracy  = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1        = f1_score(y_val, y_pred, average='weighted', zero_division=0)

    return {
        "Model": "KNN",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Train Time (s)": train_time
    }, knn

In [ ]:
!pip install faiss-cpu


In [ ]:
import faiss
from collections import Counter
from sklearn.utils import resample
from collections import defaultdict

In [ ]:

class FAISSKNNClassifier:
    def __init__(self, index, y_ref, k=6, batch_size=50_000):
        self.index = index
        self.y_ref = y_ref
        self.k = k
        self.batch_size = batch_size

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        preds = []

        for i in range(0, len(X), self.batch_size):
            X_batch = X[i:i + self.batch_size]
            _, I = self.index.search(X_batch, self.k)

            for neighbors in I:
                labels = self.y_ref[neighbors]
                preds.append(np.bincount(labels).argmax())

        return np.array(preds)

def train_faiss_knn(
    X_train, X_val, y_train, y_val,
    n_samples=200_000,
    n_neighbors=6,
    batch_size=50_000,
    random_state=42
):
    print("FAISS KNN training...")
    # --- stratified sampling ---
    X_sub, y_sub = resample(
        X_train,
        y_train,
        n_samples=n_samples,
        replace=False,
        stratify=y_train,
        random_state=random_state
    )

    dim = X_sub.shape[1]
    index = faiss.IndexFlatL2(dim)

    start = time.time()
    index.add(X_sub)
    train_time = round(time.time() - start, 4)

    # --- validation prediction ---
    y_pred = []
    for i in range(0, len(X_val), batch_size):
        X_batch = X_val[i:i + batch_size]
        _, I = index.search(X_batch, n_neighbors)

        for neighbors in I:
            labels = y_sub[neighbors]
            y_pred.append(np.bincount(labels).argmax())

    y_pred = np.array(y_pred)

    val_result = {
        "Model": f"FAISS-KNN (sampled={n_samples})",
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
        "Train Time (s)": train_time,
    }

    model = FAISSKNNClassifier(
        index=index,
        y_ref=y_sub,
        k=n_neighbors,
        batch_size=batch_size
    )

    return val_result, model


In [ ]:
class FAISSHNSWKNNClassifier:
    def __init__(self, index, y_ref, k=7, batch_size=200_000):
        self.index = index
        self.y_ref = y_ref
        self.k = k
        self.batch_size = batch_size

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        preds = []

        for i in range(0, len(X), self.batch_size):
            X_batch = X[i:i + self.batch_size]

            # search neighbors + distance
            D, I = self.index.search(X_batch, self.k)

            for dists, neighbors in zip(D, I):
                labels = self.y_ref[neighbors]

                # ----- distance-weighted voting -----
                weights = 1.0 / (dists + 1e-6)
                score = defaultdict(float)

                for lbl, w in zip(labels, weights):
                    score[lbl] += w

                pred = max(score, key=score.get)
                preds.append(pred)

        return np.array(preds)

def train_faiss_hnsw_knn(
    X_train, X_val, y_train, y_val,
    n_samples=150_000,
    n_neighbors=7,
    M=32,
    efSearch=64,
    batch_size=200_000,
    random_state=42
):
    print("FAISS HNSW-KNN training...")
    # --- stratified sampling ---
    X_sub, y_sub = resample(
        X_train, y_train,
        n_samples=n_samples,
        replace=False,
        stratify=y_train,
        random_state=random_state
    )

    dim = X_sub.shape[1]

    # --- build HNSW index ---
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efSearch = efSearch

    start = time.time()
    index.add(X_sub)
    train_time = round(time.time() - start, 4)

    # --- validation ---
    preds = []
    for i in range(0, len(X_val), batch_size):
        X_batch = X_val[i:i + batch_size]
        D, I = index.search(X_batch, n_neighbors)

        for dists, neighbors in zip(D, I):
            labels = y_sub[neighbors]
            weights = 1.0 / (dists + 1e-6)

            score = defaultdict(float)
            for lbl, w in zip(labels, weights):
                score[lbl] += w

            preds.append(max(score, key=score.get))

    y_pred = np.array(preds)

    val_result = {
        "Model": f"FAISS-HNSW-KNN-W (sampled={n_samples})",
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
        "Train Time (s)": train_time
    }

    model = FAISSHNSWKNNClassifier(
        index=index,
        y_ref=y_sub,
        k=n_neighbors,
        batch_size=batch_size
    )

    return val_result, model



In [ ]:

class FAISSHNSWKNNHybridClassifier:
    def __init__(
        self,
        index,
        y_ref,
        k=5,
        batch_size=200_000,
        majority_threshold=0.6
    ):
        self.index = index
        self.y_ref = y_ref
        self.k = k
        self.batch_size = batch_size
        self.majority_threshold = majority_threshold

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        preds = []

        for i in range(0, len(X), self.batch_size):
            X_batch = X[i:i + self.batch_size]

            # search neighbors + distances
            D, I = self.index.search(X_batch, self.k)

            for dists, neighbors in zip(D, I):
                labels = self.y_ref[neighbors]
                cnt = Counter(labels)

                top_label, top_count = cnt.most_common(1)[0]

                # ---- FAST PATH: clear majority ----
                if top_count / self.k >= self.majority_threshold:
                    preds.append(top_label)
                else:
                    # ---- SAFE PATH: weighted vote ----
                    weights = 1.0 / (dists + 1e-6)
                    score = defaultdict(float)

                    for lbl, w in zip(labels, weights):
                        score[lbl] += w

                    preds.append(max(score, key=score.get))

        return np.array(preds)

def train_faiss_hnsw_knn_hybrid(
    X_train, X_val, y_train, y_val,
    n_samples=150_000,
    n_neighbors=5,
    M=32,
    efSearch=48,
    batch_size=200_000,
    majority_threshold=0.6,
    random_state=42
):
    print("FAISS HNSW-KNN Hybrid training...")
    # --- stratified sampling ---
    X_sub, y_sub = resample(
        X_train, y_train,
        n_samples=n_samples,
        replace=False,
        stratify=y_train,
        random_state=random_state
    )

    dim = X_sub.shape[1]

    # --- build HNSW index ---
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efSearch = efSearch

    start = time.time()
    index.add(X_sub)
    train_time = round(time.time() - start, 4)

    # --- validation ---
    preds = []
    for i in range(0, len(X_val), batch_size):
        X_batch = X_val[i:i + batch_size]
        D, I = index.search(X_batch, n_neighbors)

        for dists, neighbors in zip(D, I):
            labels = y_sub[neighbors]
            cnt = Counter(labels)
            top_label, top_count = cnt.most_common(1)[0]

            if top_count / n_neighbors >= majority_threshold:
                preds.append(top_label)
            else:
                weights = 1.0 / (dists + 1e-6)
                score = defaultdict(float)
                for lbl, w in zip(labels, weights):
                    score[lbl] += w
                preds.append(max(score, key=score.get))

    y_pred = np.array(preds)

    val_result = {
        "Model": f"FAISS-HNSW-KNN-Hybrid (sampled={n_samples})",
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
        "Train Time (s)": train_time
    }

    model = FAISSHNSWKNNHybridClassifier(
        index=index,
        y_ref=y_sub,
        k=n_neighbors,
        batch_size=batch_size,
        majority_threshold=majority_threshold
    )

    return val_result, model


In [ ]:
class FAISSHNSWKNNHybridClassifier:
    def __init__(
        self,
        index,
        y_ref,
        k=5,
        batch_size=200_000,
        majority_threshold=0.6,
        attack_label=0,
        attack_dist_ratio=1.15,   # <--- ปรับแรง/อ่อน ได้
        min_attack_neighbors=1
    ):
        self.index = index
        self.y_ref = y_ref
        self.k = k
        self.batch_size = batch_size
        self.majority_threshold = majority_threshold
        self.attack_label = attack_label
        self.attack_dist_ratio = attack_dist_ratio
        self.min_attack_neighbors = min_attack_neighbors

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        preds = []

        for i in range(0, len(X), self.batch_size):
            X_batch = X[i:i + self.batch_size]
            D, I = self.index.search(X_batch, self.k)

            for dists, neighbors in zip(D, I):
                labels = self.y_ref[neighbors]
                cnt = Counter(labels)

                # -------------------------------
                # 1️⃣ ATTACK PRIORITY (ลด FN)
                # -------------------------------
                attack_mask = (labels == self.attack_label)
                if attack_mask.sum() >= self.min_attack_neighbors:
                    min_attack_dist = dists[attack_mask].min()
                    min_overall_dist = dists.min()

                    if min_attack_dist <= self.attack_dist_ratio * min_overall_dist:
                        preds.append(self.attack_label)
                        continue

                # -------------------------------
                # 2️⃣ FAST PATH: majority
                # -------------------------------
                top_label, top_count = cnt.most_common(1)[0]
                if top_count / self.k >= self.majority_threshold:
                    preds.append(top_label)
                    continue

                # -------------------------------
                # 3️⃣ SAFE PATH: weighted vote
                # -------------------------------
                weights = 1.0 / (dists + 1e-6)
                score = defaultdict(float)
                for lbl, w in zip(labels, weights):
                    score[lbl] += w

                preds.append(max(score, key=score.get))

        return np.array(preds)

def train_faiss_hnsw_knn_hybrid(
    X_train, X_val, y_train, y_val,
    n_samples=150_000,
    n_neighbors=5,
    M=32,
    efSearch=48,
    batch_size=200_000,
    majority_threshold=0.6,
    random_state=42
):
    print("FAISS HNSW-KNN Hybrid training...")
    # --- stratified sampling ---
    X_sub, y_sub = resample(
        X_train, y_train,
        n_samples=n_samples,
        replace=False,
        stratify=y_train,
        random_state=random_state
    )

    dim = X_sub.shape[1]

    # --- build HNSW index ---
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efSearch = efSearch

    start = time.time()
    index.add(X_sub)
    train_time = round(time.time() - start, 4)

    # --- validation ---
    preds = []
    for i in range(0, len(X_val), batch_size):
        X_batch = X_val[i:i + batch_size]
        D, I = index.search(X_batch, n_neighbors)

        for dists, neighbors in zip(D, I):
            labels = y_sub[neighbors]
            cnt = Counter(labels)
            top_label, top_count = cnt.most_common(1)[0]

            if top_count / n_neighbors >= majority_threshold:
                preds.append(top_label)
            else:
                weights = 1.0 / (dists + 1e-6)
                score = defaultdict(float)
                for lbl, w in zip(labels, weights):
                    score[lbl] += w
                preds.append(max(score, key=score.get))

    y_pred = np.array(preds)

    val_result = {
        "Model": f"FAISS-HNSW-KNN-Hybrid (sampled={n_samples})",
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
        "Train Time (s)": train_time
    }

    model = FAISSHNSWKNNHybridClassifier(
        index=index,
        y_ref=y_sub,
        k=n_neighbors,
        batch_size=batch_size,
        majority_threshold=majority_threshold,
        attack_label=0,            # ATTACK = 0
        # attack_dist_ratio=1.15,    # แนะนำช่วง 1.1 – 1.25
        attack_dist_ratio=1.25,    # แนะนำช่วง 1.1 – 1.25
        min_attack_neighbors=1
    )

    return val_result, model


#### KNN FAISS

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

In [ ]:
class FAISSHNSWKNNFastRecall(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        index=None,
        y_ref=None,
        k=7,
        batch_size=200_000,
        attack_cnt_threshold=2,
        attack_dist_threshold=0.6,
        eps=1e-6
    ):
        # เก็บพารามิเตอร์ลง self ตรงๆ ห้ามเปลี่ยนชื่อ (เพื่อให้ get_params ทำงานได้)
        self.index = index
        self.y_ref = y_ref
        self.k = k
        self.batch_size = batch_size
        self.attack_cnt_threshold = attack_cnt_threshold
        self.attack_dist_threshold = attack_dist_threshold
        self.eps = eps
        # ระบุคลาสให้ scikit-learn ทราบ
        self.classes_ = np.array([0, 1])

    def fit(self, X, y=None):
        # Stacking จะเรียกใช้ fit() ดังนั้นต้องมีฟังก์ชันนี้ไว้ (แม้ index จะถูกสร้างมาแล้ว)
        return self

    def _predict_one(self, dists, neighbors):
        labels = self.y_ref[neighbors]
        attack_mask = (labels == 1)
        attack_cnt = np.sum(attack_mask)

        if attack_cnt >= 1:
            if np.min(dists[attack_mask]) < self.attack_dist_threshold:
                return 1
        if attack_cnt >= self.attack_cnt_threshold:
            return 1
        if attack_cnt == 0:
            return 0

        weights = 1.0 / (dists + self.eps)
        score = {0: 0.0, 1: 0.0}
        for lbl, w in zip(labels, weights):
            score[lbl] += w
        return max(score, key=score.get)

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        preds = []
        k_int = int(self.k)
        for i in range(0, len(X), self.batch_size):
            X_batch = X[i:i + self.batch_size]
            num_samples = len(X_batch)
            D = np.empty((num_samples, k_int), dtype=np.float32)
            I = np.empty((num_samples, k_int), dtype=np.int64)
            self.index.search(X_batch, k_int)
            for dists, neighbors in zip(D, I):
                preds.append(self._predict_one(dists, neighbors))
        return np.array(preds, dtype=np.int64)

    def predict_proba(self, X):
          X = np.ascontiguousarray(X, dtype=np.float32)
          n_queries = len(X)
          probs = []
          k_int = int(self.k)

          for i in range(0, n_queries, self.batch_size):
              X_batch = np.ascontiguousarray(X[i:i + self.batch_size], dtype=np.float32)
              num_samples = len(X_batch)

              # สร้าง Buffer เตรียมไว้
              D = np.empty((num_samples, k_int), dtype=np.float32)
              I = np.empty((num_samples, k_int), dtype=np.int64)

              # ลองเรียกโดยระบุชื่อ Parameter เพื่อบังคับให้ตรงกับ Signature ของมัน
              # บางเวอร์ชันใช้ 'distances' และ 'labels' แทน 'D' และ 'I'
              try:
                  # วิธีที่ 1: ลองแบบมาตรฐานที่ใส่ Buffer
                  self.index.search(X_batch, k_int, D, I)
              except TypeError:
                  # วิธีที่ 2: หากวิธีแรกไม่ได้ ให้ระบุชื่อ keyword (สำหรับบาง Wrapper)
                  self.index.search(x=X_batch, k=k_int, distances=D, labels=I)

              for neighbors in I:
                  labels = self.y_ref[neighbors]
                  attack_prob = np.mean(labels == 1)
                  probs.append([1.0 - attack_prob, attack_prob])

          return np.array(probs)

def train_faiss_hnsw_knn_fast_recall(
    X_train, X_val, y_train, y_val,
    n_samples=300_000,
    k=7,
    M=160,
    efSearch=512,
    batch_size=300_000,
    attack_cnt_threshold=2,
    attack_dist_threshold=0.6,
    random_state=42
):
    print("FAISS HNSW-KNN Fast Recall training...")

    X_train = np.asarray(X_train, dtype=np.float32)
    X_val   = np.asarray(X_val,   dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int64)
    y_val   = np.asarray(y_val,   dtype=np.int64)

    # ---- stratified sampling ----
    X_sub, y_sub = resample(
        X_train, y_train,
        n_samples=n_samples,
        replace=False,
        stratify=y_train,
        random_state=random_state
    )

    dim = X_sub.shape[1]

    # ---- HNSW Index ----
    index = faiss.IndexHNSWFlat(dim, M)
    index.hnsw.efSearch = efSearch

    start = time.time()
    index.add(X_sub)
    train_time = round(time.time() - start, 4)

    model = FAISSHNSWKNNFastRecall(
        index=index,
        y_ref=y_sub,
        k=k,
        batch_size=batch_size,
        attack_cnt_threshold=attack_cnt_threshold,
        attack_dist_threshold=attack_dist_threshold
    )

    # ---- validation ----
    start_pred = time.time()
    y_pred = model.predict(X_val)
    pred_time = round(time.time() - start_pred, 2)

    val_result = {
        "Model": f"FAISS-HNSW-KNN-FastRecall (sampled={n_samples})",
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred, zero_division=0),
        "Train Time (s)": train_time,
        "Val Predict Time (s)": pred_time
    }

    return val_result, model


### XGB

In [ ]:
def train_xgboost(X_train, X_val, y_train, y_val):
    print("[XGBoost] Training...")

    xgb = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        eval_metric='mlogloss',
        use_label_encoder=False,
        tree_method='hist'
    )

    # xgb = XGBClassifier(
        # n_estimators=150,
        # max_depth=5,
        # learning_rate=0.05,
        # eval_metric='mlogloss',
        # use_label_encoder=False,
        # tree_method='hist'
    # )
    start_time = time.time()
    xgb.fit(X_train, y_train)
    train_time = round(time.time() - start_time, 4)
    y_pred = xgb.predict(X_val)

    accuracy  = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1        = f1_score(y_val, y_pred, average='weighted', zero_division=0)

    return {
        "Model": "XGBoost (100 trees, depth=3, lr=0.1, mlogloss)",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Train Time (s)": train_time
    }, xgb

### LR

In [ ]:
def train_logistic_regression(X_train, X_val, y_train, y_val):
    lr = LogisticRegression(
        class_weight={0: 1, 1: 15},
        solver='sag',
        max_iter=5000,
        n_jobs=-1,
        random_state=42
    )

    print("[Logistic Regression] Training...")
    start = time.time()
    lr.fit(X_train, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = lr.predict(X_val)

    accuracy  = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1        = f1_score(y_val, y_pred, average='weighted', zero_division=0)

    return {
        "Model": "Logistic Regression",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Train Time (s)": train_time
    }, lr

### DT

In [ ]:
def train_decision_tree(X_train, X_val, y_train, y_val):
    # dt = DecisionTreeClassifier(random_state=42)
    dt = DecisionTreeClassifier(random_state=42,max_depth=10, min_samples_leaf=5)

    print("[Decision Tree] Training...")
    start = time.time()
    dt.fit(X_train, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = dt.predict(X_val)

    accuracy  = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    recall    = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1        = f1_score(y_val, y_pred, average='weighted', zero_division=0)

    return {
        "Model": "Decision Tree",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Train Time (s)": train_time
    }, dt

### SVM

In [ ]:
def train_svm(X_train, X_val, y_train, y_val):
    svm_model = LinearSVC(
        class_weight={0: 1, 1: 18},
        dual=False,
        max_iter=5000,
        random_state=42
    )

    print("[SVM] Training...")
    start = time.time()
    svm_model.fit(X_train, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = svm_model.predict(X_val)

    accuracy  = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall    = recall_score(y_val, y_pred, zero_division=0)
    f1        = f1_score(y_val, y_pred, zero_division=0)

    return {
        "Model": "SVM",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Train Time (s)": train_time
    }, svm_model

## Compare model function

In [ ]:
def compare_models_with_test(X_train, X_val, X_test, y_train, y_val, y_test):

    models = {
        # "Random Forest": train_random_forest,
        # "KNN": train_faiss_hnsw_knn_fast_recall,
        # "XGBoost": train_xgboost,
        "Logistic Regression": train_logistic_regression,
        # "Decision Tree": train_decision_tree,
        # "SVM-0.1": train_svm_with_threshold,
        # "SVM-0.2": train_svm_with_threshold,
        # "SVM": train_svm
    }
    val_rows, test_rows = [], []
    trained_models = {}

    for name, trainer in models.items():
        val_result, model = trainer(X_train, X_val, y_train, y_val)
        val_rows.append(val_result)

        trained_models[name] = model

        train_time = val_result.get("Train Time (s)", None)
        start_pred = time.time()

        preds = model.predict(X_test)

        pred_time = time.time() - start_pred
        acc  = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec  = recall_score(y_test, preds,  zero_division=0)
        f1   = f1_score(y_test, preds, zero_division=0)
        row_test = {
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "Predict Time (s)": pred_time,
        }
        test_rows.append(row_test)
    df_test = pd.DataFrame(test_rows)
    df_val  = pd.DataFrame(val_rows)

    return df_val, df_test, trained_models


In [ ]:
def compare_models_with_test_by_models(X_test, y_test, models):
    val_rows, test_rows = [], []

    for name, model in models.items():
        start_pred = time.time()

        preds = model.predict(X_test)

        pred_time = time.time() - start_pred

        acc  = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec  = recall_score(y_test, preds,  zero_division=0)
        f1   = f1_score(y_test, preds, zero_division=0)
        row_test = {
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "Predict Time (s)": pred_time,
        }
        test_rows.append(row_test)
    df_test = pd.DataFrame(test_rows)

    return df_test


In [ ]:
def compare_miss_rate(trained_models, X_test, y_test):
    comparison_data = []

    for name in trained_models:
        model = trained_models[name]
        preds = model.predict(X_test)

        # สร้าง Confusion Matrix
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

        missed_attacks = fn
        total_attacks = tp + fn
        miss_rate = (missed_attacks / total_attacks) * 100

        comparison_data.append({
            "Model": name,
            "Missed Attacks (FN)": missed_attacks,
            "False Alarms (FP)": fp,
            "Correct Attacks (TP)": tp,
            "Correct benign (TN)": tn,
            "Attack Detection Rate (%)": 100 - miss_rate,
            "Miss Rate (%)": miss_rate
        })

    df_compare = pd.DataFrame(comparison_data)

    return df_compare.style.highlight_min(subset=['Missed Attacks (FN)'], color='lightgreen')

## Run training

### read all

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# read all
pathTrain = '/content/drive/MyDrive/Dataset492/1-12 (training set)/selected_features/scaled_featured_train.parquet'
pathTest = '/content/drive/MyDrive/Dataset492/3-11 (testing set)/selected_features/scaled_featured_test.parquet'

train_df_pd = pd.read_parquet(pathTrain, engine='pyarrow')
test_df_pd = pd.read_parquet(pathTest, engine='pyarrow')

print("Training shape:", train_df_pd.shape)
print("Training shape:", test_df_pd.shape)

X_train = train_df_pd.drop("Label", axis=1).astype(np.float32)
y_train = train_df_pd["Label"].astype(np.int32)

X_test = test_df_pd.drop("Label", axis=1).astype(np.float32)
y_test = test_df_pd["Label"].astype(np.int32)

print("Splitting train/val...")
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"Train shape: {X_train.shape}")
print(f"Val shape:   {X_val.shape}")
print(f"Test shape:  {X_test.shape}")

In [ ]:
X_train.info()

In [ ]:
train_df_pd.drop(columns="Label", inplace=True)
corr_matrix = train_df_pd.corr(method='spearman').abs()
# Calculate feature importance based on correlation with other features
# Use the sum of absolute correlations (excluding self-correlation) as importance metric
feature_importance = corr_matrix.sum(axis=0) - 1  # Subtract 1 to exclude self-correlation (which is 1.0)

# Sort features by importance
feature_importance_sorted = feature_importance.sort_values(ascending=False)

# Calculate percentage importance
total_importance = feature_importance_sorted.sum()
importance_percentage = (feature_importance_sorted / total_importance) * 100

# Create color gradient based on importance percentage
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Normalize the percentage values to [0, 1] for color mapping
norm = mcolors.Normalize(vmin=importance_percentage.min(), vmax=importance_percentage.max())
colors = cm.RdYlGn(norm(importance_percentage.values))  # Red (low) -> Yellow -> Green (high)

# Create bar plot
plt.figure(figsize=(12, 8))
plt.barh(range(len(importance_percentage)), importance_percentage.values, color=colors)
plt.yticks(range(len(importance_percentage)), importance_percentage.index)
plt.xlabel('Importance Percentage (%)')
plt.ylabel('Features')
plt.title('Feature Importance Based on Spearman Correlation Analysis')
plt.gca().invert_yaxis()  # Highest importance at top
plt.grid(axis='x', alpha=0.3)

# Add colorbar to show the gradient scale
sm = cm.ScalarMappable(cmap=cm.RdYlGn, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=plt.gca(), label='Importance %')

plt.tight_layout()
plt.show()

# Print the importance percentages
print("\nFeature Importance Percentages:")
for feature, percentage in importance_percentage.items():
    print(f"{feature}: {percentage:.2f}%")

### train/test

In [ ]:
df_val, df_test, trained_models = compare_models_with_test(
    X_train, X_val, X_test,
    y_train, y_val, y_test
)

print("\n--- Validation Results ---")
display(df_val.style.background_gradient(cmap="Blues"))
print("\n--- Test Results ---")
display(df_test.style.background_gradient(cmap="Blues"))

In [ ]:
model = {}
model["LR"] = grid_search
df_miss_analysis = compare_miss_rate(model, X_test, y_test)
df_miss_analysis

In [ ]:
df_test = compare_models_with_test_by_models(X_test, y_test, models)
print("\n--- Test Results ---")
display(df_test.style.background_gradient(cmap="Blues"))

### stacking model

In [ ]:
def train_parallel_stacking(X_train, y_train, models, n_splits=3):
    """
    X_train, y_train: ข้อมูลที่ใช้เทรน
    models: dictionary ที่เก็บ 'svm', 'knn', และ 'lr' (ตัวที่เราจูนมาแล้ว)
    n_splits: จำนวน Fold ในการทำ CV (แนะนำที่ 3 สำหรับข้อมูลขนาดใหญ่)
    """

    # ดึงตัวแปรกองหน้า (Base Estimators)
    estimator_list = [
        ('svm', models['svm']),
        ('knn', models['knn'])
    ]

    # สร้าง StackingClassifier พร้อมตั้งค่าขนาน
    stack_model = StackingClassifier(
        estimators=estimator_list,
        final_estimator=models['lr'],
        n_jobs=-1,
        cv=n_splits,
    )

    print(f"🚀 [Parallel Stacking] Training with {n_splits}-Fold CV...")
    start = time.time()
    print("[stacking] Training...")

    stack_model.fit(X_train, y_train)

    train_time = time.time() - start
    print(f"✅ Training Complete! Total time: {train_time:.2f} s")

    return stack_model, train_time

In [ ]:
stack_model, train_time = train_parallel_stacking(X_train, y_train, models)

y_train_pred = stack_model.predict(X_train)

In [ ]:
accuracy  = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
recall    = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
f1        = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

df_train = pd.DataFrame({
    "Model": ["Stacked (SVM + KNN)"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1 Score": [f1],
    "Train Time (s)": [train_time]
})

print("\n--- Stacking Result (On Train Set) ---")
display(df_train.style.background_gradient(cmap="Blues"))

In [ ]:
start = time.time()
# y_test_pred = stack_model.predict(X_test)
y_test_pred = stack_model.predict_proba(X_test)
test_time = round(time.time() - start, 4)

accuracy  = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
f1        = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

df_test = pd.DataFrame({
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1 Score": [f1],
    "Test Time (s)": [test_time]
})

print("\n--- Test Results ---")
display(df_test.style.background_gradient(cmap="Blues"))

In [ ]:
trained_model = {'stack_model':stack_model}
df_miss_analysis = compare_miss_rate(trained_model, X_test, y_test)
df_miss_analysis

### save model

In [ ]:
for i in range(1):
  fileName = f'LRv2.pkl'
  modelName = f'Logistic Regression'

  # --- การ Save Model ---
  path = '/content/drive/MyDrive/ml_models/LR/'+fileName
  f = open(path, 'wb')
  joblib.dump(trained_model[modelName], f)
  # joblib.dump(stack_model, f)
  f.close()
  print(f"Model saved to {path}")

In [ ]:
import faiss

model = trained_models['KNN']
path = '/content/drive/MyDrive/ml_models/KNN/'

# 1. บันทึก Index ของ FAISS แยกเป็นไฟล์ (แนะนำเพื่อความเสถียรของหน่วยความจำ)
faiss.write_index(model.index, path+"knn_index3k-k7-M128.faiss")

# 2. บันทึกส่วนที่เหลือ (y_ref, k, thresholds) ไว้ใน joblib
# เราจะเอา index ออกชั่วคราวเพื่อไม่ให้ joblib บวมหรือพัง
temp_index = model.index
model.index = None

joblib.dump(model, path+"KNN-FAISS3k-k7-M128.pkl")
model.index = temp_index

### load model

In [ ]:
models = {}

fileName = ['SVM']
modelName = ['svm']
folder = ['SVM']

for i in range(len(fileName)):
  path = '/content/drive/MyDrive/ml_models/' + folder[i] + '/' + fileName[i] + '.pkl'

  f = open(path, 'rb')
  loaded_model = joblib.load(f)
  models[modelName[i]] = loaded_model
  f.close()

  print(f"Model {modelName[i]} loaded successfully")

In [ ]:
import faiss

path = '/content/drive/MyDrive/ml_models/KNN/'
loaded_model = joblib.load(path + "KNN-FAISS3k-k7-M128.pkl")
loaded_model.index = faiss.read_index(path + "knn_index3k-k7-M128.faiss")

models['knn'] = loaded_model

In [ ]:
path = '/content/drive/MyDrive/ml_models/STACKING/'
stack_model = joblib.load(path + "KNN-SVM-LRv2.pkl")
models["stacking"] = stack_model
print(f"Stacking loaded successfully")

### confusion matrix

In [ ]:
model_name = 'LR'
# best_model = trained_models[model_name]
best_model = grid_search

print(f"Generating predictions for {model_name} on the test set...")
start_time = time.time()

y_pred_test = best_model.predict(X_test)
# decision_scores = trained_models[model_name].decision_function(X_test)
# y_pred_test = (decision_scores > t).astype(int)

prediction_time = round(time.time() - start_time, 4)
print(f"Prediction time: {prediction_time} seconds")

# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred_test)

label = ['BINIGN', 'Attack']
# Plot the confusion matrix
plt.figure(figsize=(8, 6))
ax = sns.heatmap(cm, annot=True,
            fmt='d',
            cmap='Blues',
            cbar=False,
            xticklabels=label,
            yticklabels=label)
ax.invert_yaxis()
ax.invert_xaxis()
plt.title(f'Confusion Matrix for {model_name}')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()